# Imports

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from extract import load_dataframes

from pyspark.sql import SparkSession

from pyspark.ml import Pipeline

from functools import reduce
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import FeatureHasher
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.sql.functions import udf
from pyspark.sql.functions import when, lit

# Data loading

In [2]:
spark = SparkSession.builder.appName("Analysis").getOrCreate()

dfs = load_dataframes(spark, "/app/Data")

df_acc = dfs["accidents"]

Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/30 11:09:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Data read:

Demographics (PUSA + PUSB):


+---+-------------+--------+-------+----+------+---+-------+-----+----+
| RT|     SERIALNO|DIVISION|SPORDER|PUMA|REGION| ST| ADJINC|PWGTP|AGEP|
+---+-------------+--------+-------+----+------+---+-------+-----+----+
|  P|2022GQ0000031|       6|      1|2803|     3| 01|1042311|   69|  85|
|  P|2022GQ0000111|       6|      1|2000|     3| 01|1042311|   22|  51|
|  P|2022GQ0000200|       6|      1|2803|     3| 01|1042311|   45|  36|
|  P|2022GQ0000261|       6|      1|1100|     3| 01|1042311|    4|  74|
|  P|2022GQ0000296|       6|      1|1502|     3| 01|1042311|   47|  49|
+---+-------------+--------+-------+----+------+---+-------+-----+----+
only showing top 5 rows

Accidents:
+---+-------+--------+-------------------+-------------------+---------+----------+-------+-------+------------+
| ID| Source|Severity|         Start_Time|           End_Time|Start_Lat| Start_Lng|End_Lat|End_Lng|Distance(mi)|
+---+-------+--------+-------------------+-------------------+---------+----------+-------

----------------------------------------
Exception occurred during processing of request from ('127.0.0.1', 59550)
Traceback (most recent call last):
  File "/usr/local/lib/python3.9/socketserver.py", line 316, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/local/lib/python3.9/socketserver.py", line 347, in process_request
    self.finish_request(request, client_address)
  File "/usr/local/lib/python3.9/socketserver.py", line 360, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/local/lib/python3.9/socketserver.py", line 747, in __init__
    self.handle()
  File "/usr/local/lib/python3.9/site-packages/pyspark/accumulators.py", line 262, in handle
    poll(accum_updates)
  File "/usr/local/lib/python3.9/site-packages/pyspark/accumulators.py", line 235, in poll
    if func():
  File "/usr/local/lib/python3.9/site-packages/pyspark/accumulators.py", line 239, in accum_updates
    num_updates = read_int(sel

# Data preprocessing

In [3]:
# Preprocessing
def preprocess_data(df):
    null_counts_before = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0].asDict()
    print("=== NULL counts before cleaning ===")
    for col, count in null_counts_before.items():
        print(f"{col}: {count}")

    mean_cols = ['Precipitation(in)', 'Wind_Chill(F)', 'Wind_Speed(mph)']
    means = df.select([F.mean(c).alias(c) for c in mean_cols]).collect()[0].asDict()
    print("\n=== Filling with mean ===")
    for c in mean_cols:
        print(f"{c}: filling {null_counts_before[c]} nulls with mean={means[c]}")
    df = df.fillna(means)

    drop_cols = ['End_Lng', 'End_Lat']
    print("\n=== Dropping columns ===")
    print("Dropped:", drop_cols)
    df = df.drop(*drop_cols)

    rows_before = df.count()
    rows_to_drop = df.filter(reduce(lambda a, b: a | b, [F.col(c).isNull() for c in df.columns])).count()
    print("\n=== Dropping remaining rows with nulls ===")
    print(f"Rows before: {rows_before}")
    print(f"Rows with remaining nulls to drop: {rows_to_drop}")
    df = df.dropna()
    rows_after = df.count()
    print(f"Rows after cleaning: {rows_after}")
    print(f"Total rows removed: {rows_before - rows_after}")

    null_counts_final = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).collect()[0].asDict()
    print("\n=== NULL counts after cleaning ===")
    for col, count in null_counts_final.items():
        print(f"{col}: {count}")

    return df

# Models

### Logistic Regression

In [4]:
# Train Logistic Regression
def train_logreg(df, use_class_weights=False, sample_fraction=0.1, num_features=512):
    """
    Training Logistic Regression (multiclass) on Spark with FeatureHasher.
    Handling null, empty strings and types inside a function.
    
    Parameters:
    df - Spark DataFrame (cleaned or not)
    use_class_weights - use class weights
    sample_fraction - subsampling (to avoid memory crunch)
    num_features - number of features for FeatureHasher
    
    Returns:
    model, train_df, test_df, predictions
    """
    
    df_sample = df.sample(withReplacement=False, fraction=sample_fraction, seed=42)
    
    high_card_cols = ['ID', 'Source', 'Start_Time', 'End_Time', 'End_Lat', 'End_Lng', 
                      'Description', 'Street', 'City', 'Zipcode', 'Country',
                      'Weather_Timestamp', 'Airport_Code', 'Turning_Loop']
    df_sample = df_sample.drop(*[c for c in high_card_cols if c in df_sample.columns])
    
    bool_cols = [c for c, t in df_sample.dtypes if t == 'boolean']
    for c in bool_cols:
        df_sample = df_sample.withColumn(c, F.col(c).cast("int"))
    
    numeric_cols = ['Start_Lat', 'Start_Lng', 'Distance(mi)', 'Temperature(F)',
                    'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
                    'Wind_Speed(mph)', 'Precipitation(in)']
    numeric_cols = [c for c in numeric_cols if c in df_sample.columns]
    
    for c in numeric_cols:
        df_sample = df_sample.withColumn(c, F.col(c).cast(DoubleType()))
    
    all_cols = df_sample.columns
    categorical_cols = [c for c in all_cols if c not in numeric_cols + ['Severity']]
    categorical_for_hashing = categorical_cols + bool_cols
    
    for c in categorical_for_hashing:
        df_sample = df_sample.withColumn(
            c,
            when(F.col(c).isNull() | (F.col(c) == ''), lit("missing"))
            .otherwise(F.col(c))
        )
    
    hasher = FeatureHasher(
        inputCols=categorical_for_hashing + numeric_cols,
        outputCol="features",
        numFeatures=num_features
    )
    
    lr_params = {
        "featuresCol": "features",
        "labelCol": "Severity",
        "maxIter": 100,
        "regParam": 0.01,
        "elasticNetParam": 0.0
    }
    
    if use_class_weights:
        counts = df_sample.groupBy("Severity").count().collect()
        total_count = sum(row['count'] for row in counts)
        num_classes = len(counts)
        class_weights = {row['Severity']: total_count / (num_classes * row['count']) for row in counts}
        weight_udf = udf(lambda s: float(class_weights[s]), DoubleType())
        df_sample = df_sample.withColumn("classWeight", weight_udf(F.col("Severity")))
        lr_params["weightCol"] = "classWeight"
    
    lr = LogisticRegression(**lr_params)
    
    pipeline = Pipeline(stages=[hasher, lr])
    
    train_df, test_df = df_sample.randomSplit([0.8, 0.2], seed=42)
    print(f"Train rows: {train_df.count()}, Test rows: {test_df.count()}")
    
    model = pipeline.fit(train_df)
    
    predictions = model.transform(test_df)
    
    return model, train_df, test_df, predictions

# Evaluate Model
def evaluate_model(model, train_df, test_df, predictions):
    evaluator = MulticlassClassificationEvaluator(labelCol="Severity", predictionCol="prediction", metricName="accuracy")

    # --- Train metrics ---
    lr_model = model.stages[-1]
    training_summary = lr_model.summary
    train_rdd = training_summary.predictions.select("prediction", "Severity").rdd.map(lambda row: (row[0], row[1]))
    train_metrics = MulticlassMetrics(train_rdd)

    print("\n=== Training Data ===")
    print("Accuracy:", training_summary.accuracy)
    print("Confusion Matrix:")
    print(train_metrics.confusionMatrix().toArray())
    labels = sorted(training_summary.predictions.select("Severity").distinct().rdd.flatMap(lambda x: x).collect())
    for label in labels:
        print(f"\nClass {label}:")
        print("Precision:", train_metrics.precision(label))
        print("Recall:", train_metrics.recall(label))
        print("F1-score:", train_metrics.fMeasure(label))

    # --- Test metrics ---
    test_rdd = predictions.select("prediction", "Severity").rdd.map(lambda row: (float(row[0]), float(row[1])))
    test_metrics = MulticlassMetrics(test_rdd)

    print("\n=== Test Data ===")
    print("Accuracy:", evaluator.evaluate(predictions))
    print("Confusion Matrix:")
    print(test_metrics.confusionMatrix().toArray())
    labels = sorted([float(x[0]) for x in predictions.select("Severity").distinct().collect()])
    for label in labels:
        print(f"\nClass {label}:")
        print("Precision:", test_metrics.precision(label))
        print("Recall:", test_metrics.recall(label))
        print("F1-score:", test_metrics.fMeasure(label))


In [5]:
df_clean = preprocess_data(df_acc)

25/11/30 11:09:39 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


=== NULL counts before cleaning ===
ID: 0
Source: 0
Severity: 0
Start_Time: 0
End_Time: 0
Start_Lat: 0
Start_Lng: 0
End_Lat: 3402762
End_Lng: 3402762
Distance(mi): 0
Description: 5
Street: 10869
City: 253
County: 0
State: 0
Zipcode: 1915
Country: 0
Timezone: 7808
Airport_Code: 22635
Weather_Timestamp: 120228
Temperature(F): 163853
Wind_Chill(F): 1999019
Humidity(%): 174144
Pressure(in): 140679
Visibility(mi): 177098
Wind_Direction: 175206
Wind_Speed(mph): 571233
Precipitation(in): 2203586
Weather_Condition: 173459
Amenity: 0
Bump: 0
Crossing: 0
Give_Way: 0
Junction: 0
No_Exit: 0
Railway: 0
Roundabout: 0
Station: 0
Stop: 0
Traffic_Calming: 0
Traffic_Signal: 0
Turning_Loop: 0
Sunrise_Sunset: 23246
Civil_Twilight: 23246
Nautical_Twilight: 23246
Astronomical_Twilight: 23246



=== Filling with mean ===
Precipitation(in): filling 2203586 nulls with mean=0.008407209775455218
Wind_Chill(F): filling 1999019 nulls with mean=58.251048395009235
Wind_Speed(mph): filling 571233 nulls with mean=7.68548959854036

=== Dropping columns ===
Dropped: ['End_Lng', 'End_Lat']



=== Dropping remaining rows with nulls ===
Rows before: 7728394
Rows with remaining nulls to drop: 301665


Rows after cleaning: 7426729
Total rows removed: 301665



=== NULL counts after cleaning ===
ID: 0
Source: 0
Severity: 0
Start_Time: 0
End_Time: 0
Start_Lat: 0
Start_Lng: 0
Distance(mi): 0
Description: 0
Street: 0
City: 0
County: 0
State: 0
Zipcode: 0
Country: 0
Timezone: 0
Airport_Code: 0
Weather_Timestamp: 0
Temperature(F): 0
Wind_Chill(F): 0
Humidity(%): 0
Pressure(in): 0
Visibility(mi): 0
Wind_Direction: 0
Wind_Speed(mph): 0
Precipitation(in): 0
Weather_Condition: 0
Amenity: 0
Bump: 0
Crossing: 0
Give_Way: 0
Junction: 0
No_Exit: 0
Railway: 0
Roundabout: 0
Station: 0
Stop: 0
Traffic_Calming: 0
Traffic_Signal: 0
Turning_Loop: 0
Sunrise_Sunset: 0
Civil_Twilight: 0
Nautical_Twilight: 0
Astronomical_Twilight: 0


In [6]:
model, train_df, test_df, predictions = train_logreg(df_clean, use_class_weights=False, sample_fraction=0.1)

Train rows: 594769, Test rows: 148971


25/11/30 11:13:07 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
25/11/30 11:13:07 WARN InstanceBuilder$NativeBLAS: Failed to load implementation from:dev.ludovic.netlib.blas.ForeignLinkerBLAS
25/11/30 11:13:32 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeSystemBLAS
25/11/30 11:13:32 WARN BLAS: Failed to load implementation from: com.github.fommil.netlib.NativeRefBLAS


In [7]:
evaluate_model(model, train_df, test_df, predictions)

/usr/local/lib/python3.9/site-packages/pyspark/sql/context.py:125: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(



=== Training Data ===


Accuracy: 0.8029554331177314
Confusion Matrix:


/usr/local/lib/python3.9/site-packages/pyspark/sql/context.py:125: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


[[0.00000e+00 5.06800e+03 2.60000e+01 0.00000e+00]
 [0.00000e+00 4.63958e+05 1.01030e+04 7.60000e+01]
 [0.00000e+00 8.63820e+04 1.35330e+04 4.40000e+01]
 [0.00000e+00 1.44870e+04 1.01000e+03 8.20000e+01]]



Class 1.0:
Precision: 0.0
Recall: 0.0
F1-score: 0.0

Class 2.0:
Precision: 0.8141113713929758
Recall: 0.9785315214800785
F1-score: 0.8887811867835468

Class 3.0:
Precision: 0.5485165369649806
Recall: 0.1353855080583039
F1-score: 0.21716908313340988

Class 4.0:
Precision: 0.40594059405940597
Recall: 0.00526349573143334
F1-score: 0.01039224383752614



=== Test Data ===


Accuracy: 0.8032972860489626
Confusion Matrix:


[[0.00000e+00 1.22000e+03 1.20000e+01 0.00000e+00]
 [0.00000e+00 1.16342e+05 2.52500e+03 1.90000e+01]
 [0.00000e+00 2.17180e+04 3.31000e+03 1.40000e+01]
 [0.00000e+00 3.54000e+03 2.55000e+02 1.60000e+01]]



Class 1.0:
Precision: 0.0
Recall: 0.0
F1-score: 0.0

Class 2.0:
Precision: 0.814605797507352
Recall: 0.9786013491916626
F1-score: 0.8891045677210306

Class 3.0:
Precision: 0.5424450999672239
Recall: 0.13217794105902084
F1-score: 0.2125610069355253

Class 4.0:
Precision: 0.32653061224489793
Recall: 0.004198373130411965
F1-score: 0.008290155440414506


In [8]:
model_w, train_df_w, test_df_w, predictions_w = train_logreg(df_clean, use_class_weights=True, sample_fraction=0.1)

Train rows: 594769, Test rows: 148971


In [9]:
evaluate_model(model_w, train_df_w, test_df_w, predictions_w)


=== Training Data ===


Accuracy: 0.584606269276518
Confusion Matrix:


[[  3986.    506.    359.    243.]
 [100412. 190323.  99632.  83770.]
 [ 12136.  14324.  57147.  16352.]
 [  1516.   2041.   2964.   9058.]]



Class 1.0:
Precision: 0.03376535366370182
Recall: 0.7824892029839027
F1-score: 0.06473721821607224

Class 2.0:
Precision: 0.9185738969275172
Recall: 0.401409297312802
F1-score: 0.5586799954794366

Class 3.0:
Precision: 0.35694119998500956
Recall: 0.5717043988035094
F1-score: 0.4394891967653743

Class 4.0:
Precision: 0.08277967154985698
Recall: 0.5814237114063804
F1-score: 0.14492568118910099



=== Test Data ===


Accuracy: 0.4359908975572427
Confusion Matrix:


[[  949.   121.    92.    70.]
 [25099. 47623. 24920. 21244.]
 [ 3052.  3683. 14197.  4110.]
 [  370.   495.   765.  2181.]]



Class 1.0:
Precision: 0.03220223956565999
Recall: 0.7702922077922078
F1-score: 0.06182007686795647

Class 2.0:
Precision: 0.9172027271676746
Recall: 0.4005770233669229
F1-score: 0.55762025197883

Class 3.0:
Precision: 0.3551558513033472
Recall: 0.5669275616963502
F1-score: 0.436723268118617

Class 4.0:
Precision: 0.07900742619090745
Recall: 0.572290737339281
F1-score: 0.1388464476699771


### Decision Tree

In [10]:
# Train Decision Tree
def train_decision_tree(df, use_class_weights=False, sample_fraction=0.1, num_features=512, max_depth=5):
    """
    Training Decision Tree (multiclass) on Spark with FeatureHasher.
    Handling null, empty strings and types inside a function.

    Parameters:
    df - Spark DataFrame (cleaned or not)
    use_class_weights - use class weights
    sample_fraction - subsampling (to avoid memory crunch)
    num_features - number of features for FeatureHasher
    max_depth - maximum tree depth

    Returns:
    model, train_df, test_df, predictions
    """

    df_sample = df.sample(withReplacement=False, fraction=sample_fraction, seed=42)
    
    high_card_cols = ['ID', 'Source', 'Start_Time', 'End_Time', 'End_Lat', 'End_Lng', 
                      'Description', 'Street', 'City', 'Zipcode', 'Country',
                      'Weather_Timestamp', 'Airport_Code', 'Turning_Loop']
    df_sample = df_sample.drop(*[c for c in high_card_cols if c in df_sample.columns])
    
    bool_cols = [c for c, t in df_sample.dtypes if t == 'boolean']
    for c in bool_cols:
        df_sample = df_sample.withColumn(c, F.col(c).cast("int"))
    
    numeric_cols = ['Start_Lat', 'Start_Lng', 'Distance(mi)', 'Temperature(F)',
                    'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
                    'Wind_Speed(mph)', 'Precipitation(in)']
    numeric_cols = [c for c in numeric_cols if c in df_sample.columns]
    
    for c in numeric_cols:
        df_sample = df_sample.withColumn(c, F.col(c).cast(DoubleType()))
    
    all_cols = df_sample.columns
    categorical_cols = [c for c in all_cols if c not in numeric_cols + ['Severity']]
    categorical_for_hashing = categorical_cols + bool_cols
    
    for c in categorical_for_hashing:
        df_sample = df_sample.withColumn(
            c,
            when(F.col(c).isNull() | (F.col(c) == ''), lit("missing"))
            .otherwise(F.col(c))
        )
    
    hasher = FeatureHasher(
        inputCols=categorical_for_hashing + numeric_cols,
        outputCol="features",
        numFeatures=num_features
    )
    
    dt_params = {
        "featuresCol": "features",
        "labelCol": "Severity",
        "maxDepth": max_depth
    }
    
    if use_class_weights:
        counts = df_sample.groupBy("Severity").count().collect()
        total_count = sum(row['count'] for row in counts)
        num_classes = len(counts)
        class_weights = {row['Severity']: total_count / (num_classes * row['count']) for row in counts}
        weight_udf = udf(lambda s: float(class_weights[s]), DoubleType())
        df_sample = df_sample.withColumn("classWeight", weight_udf(F.col("Severity")))
        dt_params["weightCol"] = "classWeight"
    
    dt = DecisionTreeClassifier(**dt_params)
    
    pipeline = Pipeline(stages=[hasher, dt])
    
    train_df, test_df = df_sample.randomSplit([0.8, 0.2], seed=42)
    print(f"Train rows: {train_df.count()}, Test rows: {test_df.count()}")
    
    model = pipeline.fit(train_df)
    
    predictions = model.transform(test_df)
    
    return model, train_df, test_df, predictions

# Evaluate Model (Decision Tree)
def evaluate_dt_model(model, train_df, test_df, predictions):
    evaluator = MulticlassClassificationEvaluator(labelCol="Severity", predictionCol="prediction", metricName="accuracy")
    
    # --- Train metrics ---
    train_pred_rdd = model.transform(train_df).select("prediction", "Severity").rdd.map(lambda row: (float(row[0]), float(row[1])))
    train_metrics = MulticlassMetrics(train_pred_rdd)
    
    print("\n=== Training Data ===")
    print("Accuracy:", evaluator.evaluate(model.transform(train_df)))
    print("Confusion Matrix:")
    print(train_metrics.confusionMatrix().toArray())
    
    labels = sorted([float(x[0]) for x in train_df.select("Severity").distinct().collect()])
    for label in labels:
        print(f"\nClass {label}:")
        print("Precision:", train_metrics.precision(label))
        print("Recall:", train_metrics.recall(label))
        print("F1-score:", train_metrics.fMeasure(label))
    
    # --- Test metrics ---
    test_pred_rdd = predictions.select("prediction", "Severity").rdd.map(lambda row: (float(row[0]), float(row[1])))
    test_metrics = MulticlassMetrics(test_pred_rdd)
    
    print("\n=== Test Data ===")
    print("Accuracy:", evaluator.evaluate(predictions))
    print("Confusion Matrix:")
    print(test_metrics.confusionMatrix().toArray())
    
    labels = sorted([float(x[0]) for x in predictions.select("Severity").distinct().collect()])
    for label in labels:
        print(f"\nClass {label}:")
        print("Precision:", test_metrics.precision(label))
        print("Recall:", test_metrics.recall(label))
        print("F1-score:", test_metrics.fMeasure(label))


In [11]:
model, train_df, test_df, predictions = train_decision_tree(
    df_clean,
    use_class_weights=False,
    sample_fraction=0.1,
    num_features=512,
    max_depth=11
)

Train rows: 594769, Test rows: 148971


25/11/30 11:26:10 WARN MemoryStore: Not enough space to cache rdd_810_2 in memory! (computed 31.3 MiB so far)
25/11/30 11:26:10 WARN BlockManager: Persisting block rdd_810_2 to disk instead.
25/11/30 11:26:10 WARN MemoryStore: Not enough space to cache rdd_810_4 in memory! (computed 31.3 MiB so far)
25/11/30 11:26:10 WARN BlockManager: Persisting block rdd_810_4 to disk instead.
25/11/30 11:26:11 WARN MemoryStore: Not enough space to cache rdd_810_1 in memory! (computed 31.3 MiB so far)
25/11/30 11:26:11 WARN BlockManager: Persisting block rdd_810_1 to disk instead.
25/11/30 11:26:11 WARN MemoryStore: Not enough space to cache rdd_810_0 in memory! (computed 47.3 MiB so far)
25/11/30 11:26:11 WARN BlockManager: Persisting block rdd_810_0 to disk instead.
25/11/30 11:26:12 WARN MemoryStore: Not enough space to cache rdd_810_4 in memory! (computed 47.7 MiB so far)
25/11/30 11:26:12 WARN MemoryStore: Not enough space to cache rdd_810_1 in memory! (computed 47.7 MiB so far)
25/11/30 11:26:2

In [12]:
evaluate_dt_model(model, train_df, test_df, predictions)


=== Training Data ===


Accuracy: 0.8091578411114231
Confusion Matrix:


[[1.50000e+02 4.89100e+03 5.30000e+01 0.00000e+00]
 [4.00000e+01 4.69853e+05 4.08200e+03 1.62000e+02]
 [3.00000e+00 8.90610e+04 1.06820e+04 2.13000e+02]
 [0.00000e+00 1.40600e+04 9.42000e+02 5.77000e+02]]



Class 1.0:
Precision: 0.7772020725388601
Recall: 0.02944640753828033
F1-score: 0.05674295441649328

Class 2.0:
Precision: 0.8130843709170827
Recall: 0.9909646368032868
F1-score: 0.8932549557890574

Class 3.0:
Precision: 0.6778348880005076
Recall: 0.10686381416380716
F1-score: 0.184621234380131

Class 4.0:
Precision: 0.6060924369747899
Recall: 0.037037037037037035
F1-score: 0.06980823906599722



=== Test Data ===


Accuracy: 0.8070698323834841
Confusion Matrix:


[[2.10000e+01 1.19800e+03 1.30000e+01 0.00000e+00]
 [2.40000e+01 1.17576e+05 1.20100e+03 8.50000e+01]
 [1.00000e+00 2.24210e+04 2.54100e+03 7.90000e+01]
 [1.00000e+00 3.47200e+03 2.46000e+02 9.20000e+01]]



Class 1.0:
Precision: 0.44680851063829785
Recall: 0.017045454545454544
F1-score: 0.0328381548084441

Class 2.0:
Precision: 0.8127354545266025
Recall: 0.9889810406608011
F1-score: 0.8922379938759946

Class 3.0:
Precision: 0.6350912271932017
Recall: 0.10146953118760482
F1-score: 0.1749819233550253

Class 4.0:
Precision: 0.359375
Recall: 0.0241406454998688
F1-score: 0.045242193262847306


In [13]:
model, train_df, test_df, predictions = train_decision_tree(
    df_clean,
    use_class_weights=True,
    sample_fraction=0.1,
    num_features=512,
    max_depth=11
)

Train rows: 594769, Test rows: 148971


25/11/30 11:33:21 WARN MemoryStore: Not enough space to cache rdd_977_2 in memory! (computed 31.3 MiB so far)
25/11/30 11:33:21 WARN BlockManager: Persisting block rdd_977_2 to disk instead.
25/11/30 11:33:21 WARN MemoryStore: Not enough space to cache rdd_977_3 in memory! (computed 31.3 MiB so far)
25/11/30 11:33:21 WARN BlockManager: Persisting block rdd_977_3 to disk instead.
25/11/30 11:33:21 WARN MemoryStore: Not enough space to cache rdd_977_7 in memory! (computed 47.3 MiB so far)
25/11/30 11:33:21 WARN BlockManager: Persisting block rdd_977_7 to disk instead.
25/11/30 11:33:21 WARN MemoryStore: Not enough space to cache rdd_977_4 in memory! (computed 47.3 MiB so far)
25/11/30 11:33:21 WARN BlockManager: Persisting block rdd_977_4 to disk instead.
25/11/30 11:33:22 WARN MemoryStore: Not enough space to cache rdd_977_6 in memory! (computed 47.3 MiB so far)
25/11/30 11:33:22 WARN BlockManager: Persisting block rdd_977_6 to disk instead.
25/11/30 11:33:23 WARN MemoryStore: Not enoug

In [14]:
evaluate_dt_model(model, train_df, test_df, predictions)


=== Training Data ===


Accuracy: 0.46718810159910823
Confusion Matrix:


[[4.33000e+03 6.10000e+01 5.89000e+02 1.14000e+02]
 [6.03880e+04 1.94125e+05 9.22440e+04 1.27380e+05]
 [1.44240e+04 6.68800e+03 6.61540e+04 1.26930e+04]
 [3.37000e+02 1.35100e+03 6.31000e+02 1.32600e+04]]



Class 1.0:
Precision: 0.054479799695517056
Recall: 0.8500196309383589
F1-score: 0.10239674600640866

Class 2.0:
Precision: 0.9599456051427865
Recall: 0.4094280766951324
F1-score: 0.5740269264092306

Class 3.0:
Precision: 0.41445200415993183
Recall: 0.6618113426504867
F1-score: 0.5097061758168095

Class 4.0:
Precision: 0.08641420164617099
Recall: 0.8511457731561718
F1-score: 0.15689893862482696



=== Test Data ===


Accuracy: 0.46207651153580226
Confusion Matrix:


[[9.7000e+02 5.7000e+01 1.6900e+02 3.6000e+01]
 [1.5319e+04 4.8342e+04 2.3282e+04 3.1943e+04]
 [3.5800e+03 1.8250e+03 1.6396e+04 3.2410e+03]
 [9.3000e+01 4.1200e+02 1.7800e+02 3.1280e+03]]



Class 1.0:
Precision: 0.04859232541829476
Recall: 0.7873376623376623
F1-score: 0.09153534019061998

Class 2.0:
Precision: 0.9546962635279248
Recall: 0.40662483387446796
F1-score: 0.5703330541168697

Class 3.0:
Precision: 0.40964397251717677
Recall: 0.6547400367382797
F1-score: 0.5039728280080533

Class 4.0:
Precision: 0.0815687910712423
Recall: 0.8207819469955392
F1-score: 0.14839061647572285


### Random Forest

In [15]:
# Train Random Forest
def train_random_forest(df, use_class_weights=False, sample_fraction=0.1, num_features=512, num_trees=50, max_depth=5):
    """
    Training Random Forest (multiclass) on Spark with FeatureHasher.
    Handling null, empty strings and types inside a function.

    Parameters:
    df - Spark DataFrame (cleaned or not)
    use_class_weights - use class weights
    sample_fraction - subsampling (to avoid memory hogs)
    num_features - number of features for FeatureHasher
    num_trees - number of trees
    max_depth - maximum depth of trees

    Returns:
    model, train_df, test_df, predictions
    """

    df_sample = df.sample(withReplacement=False, fraction=sample_fraction, seed=42)
    
    high_card_cols = ['ID', 'Source', 'Start_Time', 'End_Time', 'End_Lat', 'End_Lng', 
                      'Description', 'Street', 'City', 'Zipcode', 'Country',
                      'Weather_Timestamp', 'Airport_Code', 'Turning_Loop']
    df_sample = df_sample.drop(*[c for c in high_card_cols if c in df_sample.columns])
    
    bool_cols = [c for c, t in df_sample.dtypes if t == 'boolean']
    for c in bool_cols:
        df_sample = df_sample.withColumn(c, F.col(c).cast("int"))
    
    numeric_cols = ['Start_Lat', 'Start_Lng', 'Distance(mi)', 'Temperature(F)',
                    'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)',
                    'Wind_Speed(mph)', 'Precipitation(in)']
    numeric_cols = [c for c in numeric_cols if c in df_sample.columns]
    
    for c in numeric_cols:
        df_sample = df_sample.withColumn(c, F.col(c).cast(DoubleType()))
    
    all_cols = df_sample.columns
    categorical_cols = [c for c in all_cols if c not in numeric_cols + ['Severity']]
    categorical_for_hashing = categorical_cols + bool_cols
    
    for c in categorical_for_hashing:
        df_sample = df_sample.withColumn(
            c,
            when(F.col(c).isNull() | (F.col(c) == ''), lit("missing"))
            .otherwise(F.col(c))
        )
    
    hasher = FeatureHasher(
        inputCols=categorical_for_hashing + numeric_cols,
        outputCol="features",
        numFeatures=num_features
    )
    
    rf_params = {
        "featuresCol": "features",
        "labelCol": "Severity",
        "numTrees": num_trees,
        "maxDepth": max_depth
    }
    
    if use_class_weights:
        counts = df_sample.groupBy("Severity").count().collect()
        total_count = sum(row['count'] for row in counts)
        num_classes = len(counts)
        class_weights = {row['Severity']: total_count / (num_classes * row['count']) for row in counts}
        weight_udf = udf(lambda s: float(class_weights[s]), DoubleType())
        df_sample = df_sample.withColumn("classWeight", weight_udf(F.col("Severity")))
        rf_params["weightCol"] = "classWeight"
    
    rf = RandomForestClassifier(**rf_params)
    
    pipeline = Pipeline(stages=[hasher, rf])
    
    train_df, test_df = df_sample.randomSplit([0.8, 0.2], seed=42)
    print(f"Train rows: {train_df.count()}, Test rows: {test_df.count()}")
    
    model = pipeline.fit(train_df)
    
    predictions = model.transform(test_df)
    
    return model, train_df, test_df, predictions

# Evaluate Model
def evaluate_rf_model(model, train_df, test_df, predictions):
    evaluator = MulticlassClassificationEvaluator(labelCol="Severity", predictionCol="prediction", metricName="accuracy")
    
    # --- Train metrics ---
    train_pred_rdd = model.transform(train_df).select("prediction", "Severity").rdd.map(lambda row: (float(row[0]), float(row[1])))
    train_metrics = MulticlassMetrics(train_pred_rdd)
    
    print("\n=== Training Data ===")
    print("Accuracy:", evaluator.evaluate(model.transform(train_df)))
    print("Confusion Matrix:")
    print(train_metrics.confusionMatrix().toArray())
    
    labels = sorted([float(x[0]) for x in train_df.select("Severity").distinct().collect()])
    for label in labels:
        print(f"\nClass {label}:")
        print("Precision:", train_metrics.precision(label))
        print("Recall:", train_metrics.recall(label))
        print("F1-score:", train_metrics.fMeasure(label))
    
    # --- Test metrics ---
    test_pred_rdd = predictions.select("prediction", "Severity").rdd.map(lambda row: (float(row[0]), float(row[1])))
    test_metrics = MulticlassMetrics(test_pred_rdd)
    
    print("\n=== Test Data ===")
    print("Accuracy:", evaluator.evaluate(predictions))
    print("Confusion Matrix:")
    print(test_metrics.confusionMatrix().toArray())
    
    labels = sorted([float(x[0]) for x in predictions.select("Severity").distinct().collect()])
    for label in labels:
        print(f"\nClass {label}:")
        print("Precision:", test_metrics.precision(label))
        print("Recall:", test_metrics.recall(label))
        print("F1-score:", test_metrics.fMeasure(label))


In [16]:
model_rf, train_df, test_df, predictions = train_random_forest(
    df_clean,
    use_class_weights=False,
    sample_fraction=0.1,
    num_features=512,
    num_trees=10,
    max_depth=18
)

Train rows: 594769, Test rows: 148971


25/11/30 11:40:06 WARN MemoryStore: Not enough space to cache rdd_1140_1 in memory! (computed 32.1 MiB so far)
25/11/30 11:40:06 WARN BlockManager: Persisting block rdd_1140_1 to disk instead.
25/11/30 11:40:06 WARN MemoryStore: Not enough space to cache rdd_1140_3 in memory! (computed 32.1 MiB so far)
25/11/30 11:40:06 WARN BlockManager: Persisting block rdd_1140_3 to disk instead.
25/11/30 11:40:06 WARN MemoryStore: Not enough space to cache rdd_1140_0 in memory! (computed 32.1 MiB so far)
25/11/30 11:40:06 WARN BlockManager: Persisting block rdd_1140_0 to disk instead.
25/11/30 11:40:06 WARN MemoryStore: Not enough space to cache rdd_1140_6 in memory! (computed 32.1 MiB so far)
25/11/30 11:40:06 WARN BlockManager: Persisting block rdd_1140_6 to disk instead.
25/11/30 11:40:08 WARN MemoryStore: Not enough space to cache rdd_1140_6 in memory! (computed 32.1 MiB so far)
25/11/30 11:40:08 WARN MemoryStore: Not enough space to cache rdd_1140_0 in memory! (computed 32.1 MiB so far)
25/11/

In [17]:
evaluate_rf_model(model_rf, train_df, test_df, predictions)

25/11/30 11:41:20 WARN DAGScheduler: Broadcasting large task binary with size 3.8 MiB



=== Training Data ===


25/11/30 11:41:31 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB
25/11/30 11:42:10 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB


Accuracy: 0.804118910030617
Confusion Matrix:


[[0.0000e+00 5.0770e+03 1.7000e+01 0.0000e+00]
 [0.0000e+00 4.7332e+05 8.1700e+02 0.0000e+00]
 [0.0000e+00 9.5018e+04 4.9410e+03 0.0000e+00]
 [0.0000e+00 1.5502e+04 7.3000e+01 4.0000e+00]]


...


=== Test Data ===


25/11/30 11:43:34 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB
25/11/30 11:44:11 WARN DAGScheduler: Broadcasting large task binary with size 3.9 MiB


Accuracy: 0.8037067617187238
Confusion Matrix:


[[0.00000e+00 1.22900e+03 3.00000e+00 0.00000e+00]
 [0.00000e+00 1.18628e+05 2.58000e+02 0.00000e+00]
 [0.00000e+00 2.39410e+04 1.10100e+03 0.00000e+00]
 [0.00000e+00 3.79400e+03 1.70000e+01 0.00000e+00]]



Class 1.0:
Precision: 0.0
Recall: 0.0
F1-score: 0.0

Class 2.0:
Precision: 0.8037563011545341
Recall: 0.9978298538095318
F1-score: 0.8903399154902093

Class 3.0:
Precision: 0.7984046410442349
Recall: 0.04396613689002476
F1-score: 0.0833427955035767

Class 4.0:
Precision: 0.0
Recall: 0.0
F1-score: 0.0


In [18]:
model_rf_weighted, train_df_w, test_df_w, predictions_w = train_random_forest(
    df_clean,
    use_class_weights=True,
    sample_fraction=0.1,
    num_features=512,
    num_trees=10,
    max_depth=18
)

Train rows: 594769, Test rows: 148971


25/11/30 11:48:04 WARN MemoryStore: Not enough space to cache rdd_1335_4 in memory! (computed 32.1 MiB so far)
25/11/30 11:48:04 WARN BlockManager: Persisting block rdd_1335_4 to disk instead.
25/11/30 11:48:04 WARN MemoryStore: Not enough space to cache rdd_1335_2 in memory! (computed 32.1 MiB so far)
25/11/30 11:48:04 WARN BlockManager: Persisting block rdd_1335_2 to disk instead.
25/11/30 11:48:04 WARN MemoryStore: Not enough space to cache rdd_1335_1 in memory! (computed 32.1 MiB so far)
25/11/30 11:48:04 WARN BlockManager: Persisting block rdd_1335_1 to disk instead.
25/11/30 11:48:04 WARN MemoryStore: Not enough space to cache rdd_1335_3 in memory! (computed 32.1 MiB so far)
25/11/30 11:48:04 WARN BlockManager: Persisting block rdd_1335_3 to disk instead.
25/11/30 11:48:05 WARN MemoryStore: Not enough space to cache rdd_1335_7 in memory! (computed 48.5 MiB so far)
25/11/30 11:48:05 WARN BlockManager: Persisting block rdd_1335_7 to disk instead.
25/11/30 11:48:07 WARN MemoryStore:

In [19]:
evaluate_rf_model(model_rf_weighted, train_df_w, test_df_w, predictions_w)

25/11/30 11:49:36 WARN DAGScheduler: Broadcasting large task binary with size 7.2 MiB



=== Training Data ===


25/11/30 11:49:43 WARN DAGScheduler: Broadcasting large task binary with size 7.3 MiB
25/11/30 11:50:15 WARN DAGScheduler: Broadcasting large task binary with size 7.3 MiB


Accuracy: 0.5262765880535132
Confusion Matrix:


[[4.59000e+03 1.78000e+02 2.25000e+02 1.01000e+02]
 [5.95770e+04 2.31917e+05 8.41270e+04 9.85160e+04]
 [1.43400e+04 1.00080e+04 6.39330e+04 1.16780e+04]
 [5.03000e+02 1.69500e+03 8.08000e+02 1.25730e+04]]


...

25/11/30 11:51:22 WARN DAGScheduler: Broadcasting large task binary with size 7.3 MiB


Accuracy: 0.5131200032221036
Confusion Matrix:


25/11/30 11:51:51 WARN DAGScheduler: Broadcasting large task binary with size 7.3 MiB


[[1.0010e+03 1.0400e+02 9.2000e+01 3.5000e+01]
 [1.5086e+04 5.7367e+04 2.1378e+04 2.5055e+04]
 [3.6470e+03 2.9650e+03 1.5378e+04 3.0520e+03]
 [1.3100e+02 6.8800e+02 2.9800e+02 2.6940e+03]]



Class 1.0:
Precision: 0.050390133400453056
Recall: 0.8125
F1-score: 0.09489500876901928

Class 2.0:
Precision: 0.9385347817551207
Recall: 0.4825378934441398
F1-score: 0.6373757013499249

Class 3.0:
Precision: 0.41398804716523985
Recall: 0.6140883316029071
F1-score: 0.4945648678201582

Class 4.0:
Precision: 0.08736541704501233
Recall: 0.7069010758331147
F1-score: 0.15551129967962593
